In [2]:
import pandas as pd

In [3]:
# Input Files
sample_inventory_file = "AlcHepNet_Sample_Inventory_2025-04-21.xlsx"
case_obs_file = "case_obs_DCC_data_release_v2-0-4.tsv"
case_rct_file = "case_rct_DCC_data_release_v2-0-4.tsv"
output_file = "aliquot_DCC_data_release_v2-0-5.tsv"

# Read dataset
sample_df = pd.read_excel(sample_inventory_file, dtype=str)
case_obs_df = pd.read_csv(case_obs_file, sep="\t", dtype=str)
case_rct_df = pd.read_csv(case_rct_file, sep="\t", dtype=str)

# Extract External Subject ID， label cohort type
def extract_subject_map(df, cohort):
    df["External Subject ID"] = df["*submitter_id"].str.extract(r"^(\d+)")
    return df[["External Subject ID", "*submitter_id"]].rename(
        columns={"*submitter_id": "case_submitter_id"}
    ).assign(cohort=cohort)

obs_map = extract_subject_map(case_obs_df, "obs")
rct_map = extract_subject_map(case_rct_df, "clinical")
subject_map = pd.concat([obs_map, rct_map], ignore_index=True).drop_duplicates("External Subject ID")

# Combine sample data and case projection
merged = sample_df.merge(subject_map, on="External Subject ID", how="left")

# Construct result DataFrame
aliquot_df = pd.DataFrame({
    "*type": ["aliquot"] * len(merged),
    "project_id": ["ARDaC-AlcHepNet"] * len(merged),
    "*submitter_id": merged["Specimen Label"],
    "*follow_ups.submitter_id": merged.apply(
        lambda row: f"{row['case_submitter_id']}_{row['Event Label'].replace('Day ', '').strip()}"
        if pd.notnull(row["case_submitter_id"]) and pd.notnull(row["Event Label"]) else None,
        axis=1
    ),
    "labs.submitter_id": merged["Visit Site"],
    "aliquot_amount": pd.NA,
    "aliquot_collection_unit": pd.NA,
    "container_type": pd.NA,
    "specimen_type": merged["Type"]
})

# 导出 TSV 文件
aliquot_df.to_csv(output_file, sep="\t", index=False)

print(f"Output：{output_file}")

Output：aliquot_DCC_data_release_v2-0-5.tsv
